In [2]:
"client_secret_184942413535-6p9p8ggo4m3g9lcd5q230kab23jf901m.apps.googleusercontent.com.json"

'client_secret_184942413535-6p9p8ggo4m3g9lcd5q230kab23jf901m.apps.googleusercontent.com.json'

In [12]:
# Install dependencies
#!pip install --quiet google-api-python-client google-auth-httplib2 google-auth-oauthlib

import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow, Flow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
CRED_FILE = "credentials.json"

def authenticate():
    if not os.path.exists(CRED_FILE):
        raise FileNotFoundError(f"{CRED_FILE} not found in current directory.")
    
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Use Flow to manually copy & paste code (console mode)
            flow = Flow.from_client_secrets_file(CRED_FILE, SCOPES, redirect_uri="urn:ietf:wg:oauth:2.0:oob")
            auth_url, _ = flow.authorization_url(prompt="consent")
            print("Please go to this URL:", auth_url)
            code = input("Enter the authorization code here: ")
            flow.fetch_token(code=code)
            creds = flow.credentials
        
        # Save credentials for next time
        with open("token.json", "w") as f:
            f.write(creds.to_json())
    
    return creds

try:
    creds = authenticate()
    service = build("drive", "v3", credentials=creds)
    about = service.about().get(fields="user(displayName,emailAddress)").execute()
    user = about.get("user", {})
    print("✅ Authentication succeeded")
    print("User name  :", user.get("displayName"))
    print("User email :", user.get("emailAddress"))

except FileNotFoundError as e:
    print("❌", e)
except HttpError as e:
    print("❌ Google Drive API error:", e)
except Exception as e:
    print("❌ Unexpected error:", e)
